# Quantization intro

This notebook focuses on post-training quantization methods for Google Gemma 3 model with 270m parameters.

Post-training model quantization is a technique that converts a pre-trained neural network from high-precision floating-point formats into lower-precision representations after training is already complete. For the validation of the methods was used an `XLSum` dataset and `c4` (Ukrainian) for calibration of GPTQ and AWQ methods. Metrics used in model evaluation - loss and ROUGE (1, 2, L).

## Installation

In [1]:
%pip install -U transformers accelerate datasets huggingface_hub llmcompressor hqq bitsandbytes rouge-score

  Using cached transformers-5.20.2-py3-none-any.whl.metadata (31 kB)
  Using cached huggingface_hub-2.3.0-py3-none-any.whl.metadata (16 kB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
import gc
import dotenv
import re
import unicodedata
import pandas as pd

from pathlib import Path
from getpass import getpass
from huggingface_hub import login
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier, QuantizationModifier
from compressed_tensors.offload import dispatch_model
from llmcompressor.modifiers.transform.awq import AWQModifier
from llmcompressor.core import reset_session
from hqq.core.quantize import BaseQuantizeConfig
from hqq.models.hf.base import AutoHQQHFModel
from rouge_score import rouge_scorer
from sklearn.model_selection import ParameterGrid

dotenv.load_dotenv()

/home/imaha/Desktop/study/small-LLM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


False

In [4]:
login(token=getpass("Hugging Face token: "), add_to_git_credential=False)


### Load XL-SUM dataset for validation

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "google/gemma-3-270m-it"
DTYPE = torch.float32
NUM_CALIBRATION_SAMPLES = 64
MAX_SEQ_LENGTH = 256

In [6]:
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

validation_dataset = load_dataset(
    "parquet",
    data_files={
        "validation": str(PROJECT_ROOT / "data" / "raw" / "xlsum" / "ukrainian" / "validation" / "0000.parquet")
    },
    split="validation",
)

val_subset = validation_dataset.shuffle(seed=42).select(range(100))
val_data = val_subset["text"]
val_summaries = val_subset["summary"]

### Rouge tokenizer for Ukrainian

In [7]:
class UnicodeWordTokenizer:
  def tokenize(self, text):
    text = unicodedata.normalize("NFC", text).lower()
    text = text.replace("’", "'").replace("ʼ", "'")
    return re.findall(r"[^\W_]+(?:'[^\W_]+)*", text)

rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=UnicodeWordTokenizer())

### Helper functions

In [8]:
def clear_memory():
    """
    Helper function to run garbage collection
    and release unused CUDA/MPS cached memory.
    """
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

def describe_model(model):
    """
    Helper function to retrieve information
    about model name and layers.
    """
    return type(model).__name__, model.model.layers

def model_size_mb(model):
    """
    Helper function to retrieve model size in MiB.
    """
    return model.get_memory_footprint() / 1_000_000

def load_baseline_model():
    """
    Helper function to load baseline model
    as a template for quantization methods.
    """
    return AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(device).eval()


def tokenize_recieve(model, prompt):
    """
    Helper function to tokenize the prompt
    and recieve an answer.
    """
    message = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(message, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)

    input_length = inputs["input_ids"].shape[1]
    new_tokens = outputs[0, input_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer

def xlsum_prompt_formation(record):
    record = re.sub(r" {2,}", " ", record)

    prompt = (
        "Підсумуй цю статтю:\n"
        f"{record}\n"
    )

    return prompt

def evaluate_model(model, tokenizer):
    """
    Helper function to evaluate model's loss
    """
    model.eval()
    device = next(model.parameters()).device

    total_loss, total_tokens = 0.0, 0

    for prompt in val_data:
        inputs = tokenizer(prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            outputs = model(
                **inputs,
                labels=inputs["input_ids"],
                use_cache=False,
            )

        token_count = inputs["input_ids"].shape[1] - 1
        total_loss += outputs.loss.item() * token_count
        total_tokens += token_count
        del outputs, inputs

    return { "loss": total_loss / total_tokens}

def evaluate_rouge(model):
    model.eval()
    rouge_res = []

    for text, reference in zip(val_data, val_summaries):
        prompt = xlsum_prompt_formation(text)
        prediction = tokenize_recieve(model, prompt)
        scores = rouge.score(target=reference, prediction=prediction)
        rouge_res.append({name: score.fmeasure for name, score in scores.items()})

    details = pd.DataFrame(rouge_res)
    rouge_averages = details[["rouge1", "rouge2", "rougeL"]].mean().to_dict()

    return rouge_averages

def model_results(baseline_model, baseline_model_loss, baseline_model_rouge, baseline_model_size, quantized_model, method_name):
    """
    Helper function for checking and comparing model's results.
    """
    example_prompt = "Explain quantization of Large Language Models in 1 sentence."

    loss_results_quantization = evaluate_model(quantized_model, tokenizer)
    loss_q = loss_results_quantization["loss"]

    rouge_results_quantization = evaluate_rouge(quantized_model)
    rouge1, rouge2, rougeL = rouge_results_quantization["rouge1"], rouge_results_quantization["rouge2"], rouge_results_quantization["rougeL"]

    print(f"\nBaseline model size (MB): {baseline_model_size:.3f}")
    print(f"Quantized {method_name} model size (MB): {model_size_mb(quantized_model):.3f}")

    print(f"Baseline loss: {baseline_model_loss:.4f}")
    print(f"{method_name} loss: {loss_q:.4f}\n")

    print("Baseline Rouge: "
          f"Rouge1: {baseline_model_rouge["rouge1"]:.4f}, "
          f"Rouge2: {baseline_model_rouge["rouge2"]:.4f}, "
          f"RougeL: {baseline_model_rouge["rougeL"]:.4f}")

    print(f"{method_name} Rouge: "
          f"Rouge1: {rouge1:.4f}, "
          f"Rouge2: {rouge2:.4f}, "
          f"RougeL: {rougeL:.4f}")

    print(f"\nPrompt: {example_prompt}")
    print(f"\nOutput Baseline: {tokenize_recieve(baseline_model, example_prompt).strip("\n")}")
    print(f"Output {method_name}: {tokenize_recieve(quantized_model, example_prompt).strip("\n")}")

## Baseline model

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(device)
model.eval()

model_type, baseline_model_descr = describe_model(model)

print(f"Model loaded on: {model.device}")
print(model_type)
print(baseline_model_descr)

del baseline_model_descr
del model
clear_memory()

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 5879.99it/s]


Model loaded on: cuda:0
Gemma3ForCausalLM
ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
      (up_proj): Linear(in_features=640, out_features=2048, bias=False)
      (down_proj): Linear(in_features=2048, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layerno

In [11]:
baseline_model = load_baseline_model()
evaluation_results_baseline = evaluate_model(baseline_model, tokenizer)
baseline_model_loss = evaluation_results_baseline["loss"]
baseline_model_rouge = evaluate_rouge(baseline_model)
baseline_size = model_size_mb(baseline_model)

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 5124.86it/s]


### Calibration Dataset preparation

In [12]:
c4_dataset = load_dataset("allenai/c4", "uk", split="train", streaming=True,)
calibration_texts = [row["text"] for row in c4_dataset.take(NUM_CALIBRATION_SAMPLES)]

encoded = tokenizer(calibration_texts, truncation=True, max_length=MAX_SEQ_LENGTH, padding=False, add_special_tokens=False)
calibration_dataset = Dataset.from_dict(dict(encoded))

print("Calibration samples:", len(calibration_dataset))
print(calibration_texts[0][:500])

Calibration samples: 64
Як обмежити швидкість Інтернету? | Комп'ютер
Відповіді на питання. > Комп'ютер > Як обмежити швидкість Інтернету?
Як обмежити швидкість Інтернету?
Насправді, суть даної дії зовсім не в тому, щоб назавжди забезпечити себе слабкими швидкостями. Це необхідно лише для того, щоб розвантажити потік: коли швидкість використовується на повну котушку, нічого іншого з Інтернетом зробити просто не можна.
Якщо Ви володієте роутером, то Ви зможете обмежити швидкість всієї мережі в цілому, так і окремих прист


## Linear symmetric and asymmetric quantization

Linear quantization is the most fundamental method of model quantization. It maps floating-point values to discrete integers using a scale factor and a zero point, reducing the precision required to represent model weights.

Symmetric quantization uses a range symmetric around zero and fixes the zero point at 0. The scale is calculated from the maximum absolute value.

Asymmetric quantization uses the minimum and maximum values to calculate both the scale and the zero point, allowing the quantization range to adapt to unevenly distributed values.

In [13]:
def symmetric_quantize(x: torch.Tensor, dtype=torch.int8) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Function implements Linear Symmetric Quantization per tensor.
    Zero point is fixed at 0.

    :param x: torch.Tensor, single tensor
    :param dtype: torch.dtype, type to downcast tensor to
    :return: tuple[torch.Tensor, torch.Tensor], downcast tensor, scale
    """
    max_original = x.abs().max()
    max_int = torch.iinfo(dtype).max
    s = max_original / max_int
    s = torch.where(s == 0, torch.ones_like(s), s)
    q = torch.round(x / s).clamp(-max_int, max_int).to(dtype)

    return q, s

def asymmetric_quantize(x: torch.Tensor, dtype=torch.int8) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Function implements Linear Asymmetric Quantization per tensor.

    :param x: torch.Tensor, single tensor
    :param dtype: torch.dtype, type to downcast tensor to
    :return: tuple[torch.Tensor, torch.Tensor, torch.Tensor], downcast tensor, scale, zero point
    """
    min_original, max_original = x.min().clamp(max=0), x.max().clamp(min=0)
    min_int, max_int = torch.iinfo(dtype).min, torch.iinfo(dtype).max

    s = (max_original - min_original) / (max_int - min_int)
    s = torch.where(s == 0, torch.ones_like(s), s)

    z = torch.round(min_int - min_original / s).clamp(min_int, max_int).to(dtype)
    q = (torch.round(x / s) + z.float()).clamp(min_int, max_int).to(dtype)

    return q, s, z

def dequantize(q: torch.Tensor, s: torch.Tensor, z: int | torch.Tensor = 0) -> torch.Tensor:
    """
    Function implements Linear Dequantization per tensor.

    :param q: torch.Tensor, quantized tensor
    :param s: torch.Tensor, scale
    :param z: torch.Tensor, zero point
    :return: torch.Tensor, dequantized tensor
    """
    return s.float() * (q.float() - torch.as_tensor(z, dtype=torch.float32, device=q.device))


class QuantizedLinear(torch.nn.Module):
    @torch.no_grad()
    def __init__(self, linear, method="symmetric"):
        super().__init__()

        self.in_features = linear.in_features
        self.out_features = linear.out_features
        weights = linear.weight.detach().float()

        if method == "symmetric":
            q, scale = symmetric_quantize(weights)
            zero_point = torch.zeros((), dtype=torch.int8, device=q.device)
        elif method == "asymmetric":
            q, scale, zero_point = asymmetric_quantize(weights)

        self.register_buffer("bias", linear.bias.detach().clone() if linear.bias is not None else None)
        self.register_buffer("qweight", q)
        self.register_buffer("scale", scale)
        self.register_buffer("zero_point", zero_point)

    def forward(self, x):
        weight = dequantize(self.qweight, self.scale, self.zero_point).to(dtype=x.dtype)
        bias = (self.bias.to(dtype=x.dtype) if self.bias is not None else None)

        return torch.nn.functional.linear(x, weight, bias)


In [14]:
@torch.no_grad()
def linear_weights_quantization(model, method="symmetric"):
    results = []

    _, model_layers = describe_model(model)

    def replace_layers(parent):
        for name, module in list(parent.named_children()):
            if isinstance(module, torch.nn.Linear):
                quantized = QuantizedLinear(module, method)

                results.append({
                    "layer": name,
                    "shape": tuple(quantized.qweight.shape),
                    "num_weights": quantized.qweight.numel(),
                    "scale": quantized.scale.item(),
                    "zero_point": quantized.zero_point.item(),
                })

                setattr(parent, name, quantized)
            else:
                replace_layers(module)

    replace_layers(model_layers)
    return results

In [15]:
symmetric_model = load_baseline_model()

results_sym = linear_weights_quantization(symmetric_model, method="symmetric",)
prompt_results_symmetric = model_results(baseline_model, baseline_model_loss, baseline_model_rouge, baseline_size, symmetric_model, "Linear Symmetric")

del symmetric_model
clear_memory()

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 6582.06it/s]



Baseline model size (MB): 1072.395
Quantized Linear Symmetric model size (MB): 771.585
Baseline loss: 3.8468
Linear Symmetric loss: 4.0384

Baseline Rouge: Rouge1: 0.1168, Rouge2: 0.0290, RougeL: 0.0914
Linear Symmetric Rouge: Rouge1: 0.1114, Rouge2: 0.0267, RougeL: 0.0848

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output Linear Symmetric: Large Language Models (LLMs) are designed to process and understand natural language, achieving a state-of-the-art performance in various tasks.


Symmetric quantization of linear layers reduced the model size by approximately 28%, with a small increase in loss and small decreases across all ROUGE scores. In the example prompt, the quantized model produced fluent text but failed to explain quantization, which suggests some degradation.

In [16]:
asymmetric_model = load_baseline_model()

results_asym = linear_weights_quantization(asymmetric_model, method="asymmetric",)
prompt_results_asymmetric = model_results(baseline_model, baseline_model_loss, baseline_model_rouge, baseline_size, asymmetric_model, "Linear Asymmetric")

del asymmetric_model
clear_memory()

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 5173.36it/s]



Baseline model size (MB): 1072.395
Quantized Linear Asymmetric model size (MB): 771.585
Baseline loss: 3.8468
Linear Asymmetric loss: 3.9352

Baseline Rouge: Rouge1: 0.1168, Rouge2: 0.0290, RougeL: 0.0914
Linear Asymmetric Rouge: Rouge1: 0.1049, Rouge2: 0.0227, RougeL: 0.0810

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output Linear Asymmetric: Quantization reduces the model's size and computational cost, making it suitable for lower-resource environments.


Asymmetric quantization shows more precise results than symmetric quantization in terms of loss, likely because it uses a zero point to shift the quantization range and better represent weight distributions that are not centered around zero, although its ROUGE scores are lower.

## NF4 model - bitsandbytes

BitsAndBytes NF4 quantization compresses model weights into a 4-bit NormalFloat format using 16 nonuniformly spaced values designed for approximately normally distributed weights. Weights are quantized in blocks with separate scaling factors and dequantized to a higher precision for computation.

In [17]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)

nf4_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quantization_config, device_map={"": 0}, dtype=DTYPE).eval()
prompt_results_nf4 = model_results(baseline_model, baseline_model_loss, baseline_model_rouge, baseline_size, nf4_model, "NF4")

del nf4_model
clear_memory()

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 1213.90it/s]



Baseline model size (MB): 1072.395
Quantized NF4 model size (MB): 721.449
Baseline loss: 3.8468
NF4 loss: 4.3888

Baseline Rouge: Rouge1: 0.1168, Rouge2: 0.0290, RougeL: 0.0914
NF4 Rouge: Rouge1: 0.1120, Rouge2: 0.0252, RougeL: 0.0847

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output NF4: Quantization of Large Language Models (LLMs) aims to reduce the size and complexity of the model by lowering the number of parameters, typically to fractions of the original size. This reduces the model's computational cost, memory requirements, and training time, leading to improved performance on specific, often less-studied, tasks.


NF4 quantization reduced the model size by approximately 32%, but increased loss from 3.8468 to 4.3888 and lowered all ROUGE scores. The example response incorrectly described quantization as reducing the number of parameters and exceeded the one-sentence instruction, indicating reduced factual accuracy and instruction following.

## GPTQ

GPTQ is a method that uses calibration data and approximate second-order information to minimize layer output error. It quantizes weights sequentially, adjusting the remaining weights to compensate for quantization errors.

In [18]:
gptq_model = load_baseline_model()
gptq_recipe = GPTQModifier(targets=["Linear"], scheme="W4A16", ignore=["lm_head"], dampening_frac=0.1, actorder=None)

oneshot(model=gptq_model, tokenizer=tokenizer, dataset=calibration_dataset, recipe=gptq_recipe,
        max_seq_length=MAX_SEQ_LENGTH,
        num_calibration_samples=len(calibration_dataset),
        batch_size=1,
        shuffle_calibration_samples=False)

dispatch_model(gptq_model)
prompt_results_gptq = model_results(baseline_model, baseline_model_loss, baseline_model_rouge, baseline_size, gptq_model, "GPTQ")

reset_session()
del gptq_model, gptq_recipe
clear_memory()

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 5168.72it/s]


2026-10-24T19:47:25.1038 | __init__ | WARNING - Disabling tokenizer parallelism due to threading conflict between FastTokenizer and Datasets. Set TOKENIZERS_PARALLELISM=false to suppress this warning.
2026-10-24T19:47:25.4932 | reset | INFO - Compression lifecycle reset
2026-10-24T19:47:25.5120 | norm_calibration_context | INFO - Found 109 offset-norm modules to convert
2026-10-24T19:47:25.5149 | from_modifiers | INFO - Creating recipe from modifiers


Applying quantization config: 100%|██████████| 126/126 [00:00<00:00, 14718.09it/s]

2026-10-24T19:47:25.5282 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-10-24T19:47:25.5284 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`



W1024 19:47:25.549000 474893 torch/fx/_symbolic_trace.py:57] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(2/19): Calibrating: 100%|██████████| 64/64 [00:00<00:00, 773.65it/s]

...

2026-10-24T19:47:28.8998 | compress_module_list | INFO - Quantizing 1 module(s): ['model.layers.17.self_attn.q_proj']
2026-10-24T19:47:28.9027 | GPTQ | METRIC - time 0.00s
2026-10-24T19:47:28.9030 | GPTQ | METRIC - error 4699.55
2026-10-24T19:47:28.9034 | GPTQ | METRIC - Accelerator 0 | usage: 17.52% | total memory: 16.7 Gb
2026-10-24T19:47:28.9041 | compress_module_list | INFO - Quantizing 2 module(s): ['model.layers.17.self_attn.k_proj', 'model.layers.17.self_attn.v_proj']
2026-10-24T19:47:28.9080 | GPTQ | METRIC - time 0.00s
2026-10-24T19:47:28.9083 | GPTQ | METRIC - error 1428.04
2026-10-24T19:47:28.9087 | GPTQ | METRIC - Accelerator 0 | usage: 17.52% | total memory: 16.7 Gb
2026-10-24T19:47:28.9089 | GPTQ | METRIC - time 0.00s
2026-10-24T19:47:28.9091 | GPTQ | METRIC - error 1529.53
2026-10-24T19:47:28.9093 | GPTQ | METRIC - Accelerator 0 | usage: 17.52% | total memory: 16.7 Gb
2026-10-24T19:47:28.9100 | compress_module_list | INFO - Quantizing 1 module(s): ['model.layers.17.self_

2026-10-24T19:47:28.9198 | GPTQ | METRIC - error 459.05
2026-10-24T19:47:28.9202 | GPTQ | METRIC - Accelerator 0 | usage: 17.52% | total memory: 16.7 Gb
2026-10-24T19:47:28.9204 | GPTQ | METRIC - time 0.00s
2026-10-24T19:47:28.9206 | GPTQ | METRIC - error 469.74
2026-10-24T19:47:28.9210 | GPTQ | METRIC - Accelerator 0 | usage: 17.52% | total memory: 16.7 Gb
2026-10-24T19:47:28.9216 | compress_module_list | INFO - Quantizing 1 module(s): ['model.layers.17.mlp.down_proj']
2026-10-24T19:47:28.9302 | GPTQ | METRIC - time 0.01s
2026-10-24T19:47:28.9306 | GPTQ | METRIC - error 6.60
2026-10-24T19:47:28.9309 | GPTQ | METRIC - Accelerator 0 | usage: 17.52% | total memory: 16.7 Gb


(19/19): Propagating: 100%|██████████| 64/64 [00:00<00:00, 1024.65it/s]

2026-10-24T19:47:28.9983 | norm_calibration_context | INFO - Restoring 109 norm modules to offset convention
2026-10-24T19:47:29.0007 | finalize | INFO - Compression lifecycle finalized for 1 modifiers



Baseline model size (MB): 1072.395
Quantized GPTQ model size (MB): 1076.312
Baseline loss: 3.8468
GPTQ loss: 4.4205

Baseline Rouge: Rouge1: 0.1168, Rouge2: 0.0290, RougeL: 0.0914
GPTQ Rouge: Rouge1: 0.0744, Rouge2: 0.0139, RougeL: 0.0587

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output GPTQ: In the field of Artificial Intelligence, quantization is a crucial technique to reduce the size of a large language model (LLM) to a manageable integer. This is done by performing operations on the model's weights and parameters, such as reducing the number of bits of information. This process ensures the model performs its intended task efficiently and accurately, ensuring the model doesn't have any off-reading or misinterpretations.

In essence, quantization is a way to make LLMs more performan

GPTQ showed a slightly larger reported model size, increased loss from 3.8468 to 4.4205, and lowered ROUGE scores. The example response contained inaccurate explanations and exceeded the one-sentence instruction, indicating reduced factual accuracy and instruction following. This GPTQ model uses a basic configuration, so tuning its quantization parameters and calibration setup may improve the results.

In [19]:
param_grid = {"dampening_frac": [0.01, 0.05, 0.1], "actorder": [None, "static"],}
gptq_search_results = []

for params in ParameterGrid(param_grid):
  print(f"\nTesting: {params}", flush=True)

  gptq_model = load_baseline_model()

  try:
    recipe = GPTQModifier(targets=["Linear"], scheme="W4A16", ignore=["lm_head"], **params,)

    oneshot(model=gptq_model, tokenizer=tokenizer, dataset=calibration_dataset,
            recipe=recipe, max_seq_length=MAX_SEQ_LENGTH,
            num_calibration_samples=len(calibration_dataset),
            batch_size=1,
            shuffle_calibration_samples=False)

    dispatch_model(gptq_model)

    loss = evaluate_model(gptq_model, tokenizer)["loss"]
    rouge_scores = evaluate_rouge(gptq_model)

    gptq_search_results.append({**params, "loss": loss, "size_MB": model_size_mb(gptq_model), **rouge_scores})

  finally:
    reset_session()
    del gptq_model, recipe
    clear_memory()

gptq_search_df = (pd.DataFrame(gptq_search_results).sort_values("loss").reset_index(drop=True))
display(gptq_search_df)


Testing: {'actorder': None, 'dampening_frac': 0.01}


Loading weights: 100%|██████████| 236/236 [00:00<00:00, 5933.21it/s]


2026-10-24T19:51:03.5969 | reset | INFO - Compression lifecycle reset
2026-10-24T19:51:03.6009 | norm_calibration_context | INFO - Found 109 offset-norm modules to convert
2026-10-24T19:51:03.6063 | from_modifiers | INFO - Creating recipe from modifiers


Applying quantization config: 100%|██████████| 126/126 [00:00<00:00, 13256.80it/s]

2026-10-24T19:51:03.6206 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-10-24T19:51:03.6209 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`


...

2026-10-24T20:03:36.4514 | GPTQ | METRIC - time 0.00s
2026-10-24T20:03:36.4516 | GPTQ | METRIC - error 511.11
2026-10-24T20:03:36.4519 | GPTQ | METRIC - Accelerator 0 | usage: 52.04% | total memory: 16.7 Gb
2026-10-24T20:03:36.4525 | compress_module_list | INFO - Quantizing 2 module(s): ['model.layers.17.mlp.gate_proj', 'model.layers.17.mlp.up_proj']
2026-10-24T20:03:36.4566 | GPTQ | METRIC - time 0.00s
2026-10-24T20:03:36.4569 | GPTQ | METRIC - error 415.77
2026-10-24T20:03:36.4573 | GPTQ | METRIC - Accelerator 0 | usage: 52.04% | total memory: 16.7 Gb
2026-10-24T20:03:36.4575 | GPTQ | METRIC - time 0.00s
2026-10-24T20:03:36.4577 | GPTQ | METRIC - error 424.81
2026-10-24T20:03:36.4580 | GPTQ | METRIC - Accelerator 0 | usage: 52.04% | total memory: 16.7 Gb
2026-10-24T20:03:36.4585 | compress_module_list | INFO - Quantizing 1 module(s): ['model.layers.17.mlp.down_proj']
2026-10-24T20:03:36.4701 | GPTQ | METRIC - time 0.01s
2026-10-24T20:03:36.4704 | GPTQ | METRIC - error 4.96
2026-10-24

(19/19): Propagating: 100%|██████████| 64/64 [00:00<00:00, 919.97it/s]

2026-10-24T20:03:36.5454 | norm_calibration_context | INFO - Restoring 109 norm modules to offset convention
2026-10-24T20:03:36.5479 | finalize | INFO - Compression lifecycle finalized for 1 modifiers


2026-10-24T20:04:34.6453 | reset | INFO - Compression lifecycle reset


,actorder,dampening_frac,loss,size_MB,rouge1,rouge2,rougeL
0,static,0.10,4.206908,1076.311556,0.030822,0.006618,0.024143
1,static,0.01,4.214457,1076.311556,0.113499,0.030978,0.087361
2,static,0.05,4.232358,1076.311556,0.006218,0.000556,0.005545
3,NaN,0.05,4.387374,1076.311556,0.000688,0.000000,0.000688
4,NaN,0.10,4.420500,1076.311556,0.074427,0.013894,0.058678
5,NaN,0.01,4.500352,1076.311556,0.013841,0.002190,0.011321


We have decided to tune the `actorder`(changes the sequence in which model weights are quantized during the compression process) and `dampening_frac`(adds regularization to the matrix calculations) parameters to improve the GPTQ results. Setting `actorder="static"` gave lower loss for all tested values of `dampening_frac`. The lowest loss of 4.2069 was achieved with `dampening_frac=0.1`, while `dampening_frac=0.01` gave the highest ROUGE scores with a similar loss 4.2145, making it a promising choice. The reported bigger model size remained the same across all configurations.

## Activation-aware Weight Quantization (AWQ)

AWQ uses activation statistics collected from calibration data to identify important weight channels and adjust their scaling to reduce quantization error.

In [20]:
awq_model = load_baseline_model()
awq_recipe = [AWQModifier(), QuantizationModifier(targets=["Linear"], scheme="W4A16_ASYM", ignore=["lm_head"])]

oneshot(model=awq_model, tokenizer=tokenizer, dataset=calibration_dataset, recipe=awq_recipe,
        max_seq_length=MAX_SEQ_LENGTH,
        num_calibration_samples=len(calibration_dataset),
        batch_size=1,
        shuffle_calibration_samples=False)

dispatch_model(awq_model)
prompt_results_awq = model_results(baseline_model, baseline_model_loss, baseline_model_rouge, baseline_size, awq_model, "AWQ")

reset_session()
del awq_model, awq_recipe
clear_memory()


Loading weights: 100%|██████████| 236/236 [00:00<00:00, 6127.66it/s]


2026-10-24T20:04:35.8273 | reset | INFO - Compression lifecycle reset
2026-10-24T20:04:35.8316 | norm_calibration_context | INFO - Found 109 offset-norm modules to convert
2026-10-24T20:04:35.8347 | from_modifiers | INFO - Creating recipe from modifiers
2026-10-24T20:04:35.8350 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...


Applying quantization config: 100%|██████████| 126/126 [00:00<00:00, 13793.45it/s]

2026-10-24T20:04:35.8484 | initialize | INFO - Compression lifecycle initialized for 2 modifiers
2026-10-24T20:04:35.8487 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `AWQModifier`
2026-10-24T20:04:35.8887 | _set_resolved_mappings | WARNING - 18 mappings were skipped due to incompatible shapes. Set logging to DEBUG to view full list.



(1/19): Calibrating: 100%|██████████| 64/64 [00:00<00:00, 1273.98it/s]
Smoothing: 0it [00:00, ?it/s]
(19/19): Propagating: 100%|██████████| 64/64 [00:00<00:00, 968.75it/s]

2026-10-24T20:04:45.2087 | IndependentPipeline | INFO - Inferred `DataFreePipeline` for `QuantizationModifier`


2026-10-24T20:04:45.2435 | norm_calibration_context | INFO - Restoring 109 norm modules to offset convention
2026-10-24T20:04:45.2471 | finalize | INFO - Compression lifecycle finalized for 2 modifiers

Baseline model size (MB): 1072.395
Quantized AWQ model size (MB): 1076.312
Baseline loss: 3.8468
AWQ loss: 4.5216

Baseline Rouge: Rouge1: 0.1168, Rouge2: 0.0290, RougeL: 0.0914
AWQ Rouge: Rouge1: 0.1068, Rouge2: 0.0283, RougeL: 0.0832

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output AWQ: Because of the limitations of the Large Language Models (LLMs) in the real world, they are designed to be used for specific tasks that are not directly related to the core functionality of the model.
2026-10-24T20:08:35.6557 | reset | INFO - Compression lifecycle reset


AWQ showed a slightly larger reported model size, increased loss from 3.8468 to 4.5216, and lowered all ROUGE scores. The example response followed the one-sentence instruction but failed to explain quantization. These results suggest that the basic AWQ configuration, as well as GPTQ model, needs further tuning.

In [21]:
param_grid = {"duo_scaling": [True, False], "n_grid": [20, 40]}
awq_search_results = []

for params in ParameterGrid(param_grid):
    print(f"\nTesting AWQ: {params}", flush=True)

    awq_model = load_baseline_model()

    try:
      recipe = [AWQModifier(**params),
                QuantizationModifier(
                targets=["Linear"],
                scheme="W4A16_ASYM",
                ignore=["lm_head"],)]

      oneshot(model=awq_model, tokenizer=tokenizer, dataset=calibration_dataset,
              recipe=recipe, max_seq_length=MAX_SEQ_LENGTH,
              num_calibration_samples=len(calibration_dataset),
              batch_size=1,
              shuffle_calibration_samples=False)

      dispatch_model(awq_model)
      loss = evaluate_model(awq_model, tokenizer)["loss"]
      rouge_scores = evaluate_rouge(awq_model)

      awq_search_results.append({**params, "loss": loss, "size_MB": model_size_mb(awq_model), **rouge_scores})

    finally:
        reset_session()
        del awq_model, recipe
        clear_memory()

awq_search_df = (pd.DataFrame(awq_search_results).sort_values("loss").reset_index(drop=True))
display(awq_search_df)


Testing AWQ: {'duo_scaling': True, 'n_grid': 20}


Loading weights: 100%|██████████| 236/236 [00:00<00:00, 6692.10it/s]


2026-10-24T20:08:36.7689 | reset | INFO - Compression lifecycle reset
2026-10-24T20:08:36.7716 | norm_calibration_context | INFO - Found 109 offset-norm modules to convert
2026-10-24T20:08:36.7744 | from_modifiers | INFO - Creating recipe from modifiers
2026-10-24T20:08:36.7749 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...


Applying quantization config: 100%|██████████| 126/126 [00:00<00:00, 10943.93it/s]

2026-10-24T20:08:36.7923 | initialize | INFO - Compression lifecycle initialized for 2 modifiers
2026-10-24T20:08:36.7926 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `AWQModifier`
2026-10-24T20:08:36.8299 | _set_resolved_mappings | WARNING - 18 mappings were skipped due to incompatible shapes. Set logging to DEBUG to view full list.


...

2026-10-24T20:17:10.9228 | initialize | INFO - Compression lifecycle initialized for 2 modifiers
2026-10-24T20:17:10.9232 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `AWQModifier`
2026-10-24T20:17:10.9678 | _set_resolved_mappings | WARNING - 18 mappings were skipped due to incompatible shapes. Set logging to DEBUG to view full list.



(1/19): Calibrating: 100%|██████████| 64/64 [00:00<00:00, 1062.49it/s]
Smoothing: 0it [00:00, ?it/s]
(19/19): Propagating: 100%|██████████| 64/64 [00:00<00:00, 885.02it/s]

2026-10-24T20:17:26.4016 | IndependentPipeline | INFO - Inferred `DataFreePipeline` for `QuantizationModifier`


2026-10-24T20:17:26.4347 | norm_calibration_context | INFO - Restoring 109 norm modules to offset convention
2026-10-24T20:17:26.4378 | finalize | INFO - Compression lifecycle finalized for 2 modifiers
2026-10-24T20:17:44.7600 | reset | INFO - Compression lifecycle reset


,duo_scaling,n_grid,loss,size_MB,rouge1,rouge2,rougeL
0,True,40,4.452035,1076.311556,0.056333,0.016977,0.043951
1,False,20,4.491760,1076.311556,0.060404,0.017165,0.045897
2,False,40,4.497848,1076.311556,0.002939,0.000000,0.001821
3,True,20,4.521560,1076.311556,0.106830,0.028279,0.083216


We have decided to tune the `duo_scaling`(controls whether scaling uses both weight and activation statistics or only activation statistics) and `n_grid`(sets the number of scaling candidates tested to find the one with the lowest quantization error) parameters to improve the AWQ results. Setting `duo_scaling=True` and `n_grid=40` gave the lowest loss of 4.4520, while the original configuration with `duo_scaling=True` and `n_grid=20` kept the highest ROUGE scores. Tuning slightly improved loss but reduced ROUGE scores, so there was no clear overall improvement. The reported model size remained the same across all configurations.

## Half-Quadratic Quantization (HQQ)

HQQ is a post-training weight quantization method that uses half-quadratic optimization to reduce the error between original and quantized weights.

In [22]:
hqq_model = load_baseline_model()
quant_config = BaseQuantizeConfig(nbits=8, group_size=64)
AutoHQQHFModel.quantize_model(hqq_model, quant_config=quant_config, compute_dtype=DTYPE, device="cuda:0",)
prompt_results_hqq = model_results(baseline_model, baseline_model_loss, baseline_model_rouge, baseline_size, hqq_model, "HQQ")

save_dir = "../models/gemma-hqq-xlsum"
AutoHQQHFModel.save_quantized(hqq_model, save_dir)
tokenizer.save_pretrained(save_dir)

del hqq_model
clear_memory()

100%|██████████| 127/127 [00:00<00:00, 905.36it/s]



Baseline model size (MB): 1072.395
Quantized HQQ model size (MB): 771.585
Baseline loss: 3.8468
HQQ loss: 3.8612

Baseline Rouge: Rouge1: 0.1168, Rouge2: 0.0290, RougeL: 0.0914
HQQ Rouge: Rouge1: 0.1136, Rouge2: 0.0277, RougeL: 0.0884

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output HQQ: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.


HQQ reduced the model size by 28%, with only a small increase in loss from 3.8468 to 3.8612 and slight decreases in all ROUGE scores. The example response was identical to the baseline, preserving both accuracy and the one-sentence instruction. These results show a good balance between size reduction and maintaining model quality.

## Results

| Experiment | Configuration | Model size (MB) | Loss | ROUGE-1 | ROUGE-2 | ROUGE-L |
|---|---|---:|---:|---:|---:|---:|
| Baseline | Original model, FP32 | 1072.395 | 3.8468 | 0.1168 | 0.0290 | 0.0914 |
| Linear Symmetric | INT8, symmetric | 771.585 | 4.0384 | 0.1114 | 0.0267 | 0.0848 |
| Linear Asymmetric | INT8, asymmetric | 771.585 | 3.9352 | 0.1049 | 0.0227 | 0.0810 |
| NF4 | 4-bit NormalFloat | 721.449 | 4.3888 | 0.1120 | 0.0252 | 0.0847 |
| GPTQ | Basic configuration | 1076.312 | 4.4205 | 0.0744 | 0.0139 | 0.0587 |
| GPTQ — lowest loss | `actorder="static"`, `dampening_frac=0.1` | 1076.312 | 4.2069 | 0.0308 | 0.0066 | 0.0241 |
| GPTQ — highest ROUGE | `actorder="static"`, `dampening_frac=0.01` | 1076.312 | 4.2145 | 0.1135 | 0.0310 | 0.0874 |
| AWQ — highest ROUGE | `duo_scaling=True`, `n_grid=20` | 1076.312 | 4.5216 | 0.1068 | 0.0283 | 0.0832 |
| AWQ — lowest loss | `duo_scaling=True`, `n_grid=40` | 1076.312 | 4.4520 | 0.0563 | 0.0170 | 0.0440 |
| HQQ | Tested configuration | 771.585 | 3.8612 | 0.1136 | 0.0277 | 0.0884 |

The experiments show that, for Gemma 3-270M evaluated on XL-Sum, Half-Quadratic Quantization (HQQ) provided the best balance between model size and quality among the tested configurations. It reduced the reported model size by 28% while keeping loss and ROUGE scores close to the baseline. GPTQ and AWQ generally performed worse in this setup, which may be related to several factors, including the calibration dataset, its size, the maximum token sequence length, and the quantization parameters. With suitable settings, quantization can reduce model size while keeping quality close to the original, as shown by the simple linear quantization approaches implemented in these experiments.